# Binding - OLMo 2 Model Family

### Olmo 2 - 1B
model_name = OLMo-2-0425-1B  
revision = stage1-step1907359-tokens4001B

### Olmo 2 - 7B
model_name = OLMo-2-1124-7B  
revision = stage1-step928646-tokens3896B

### Olmo 2 - 13B
model_name = OLMo-2-1124-13B  
revision = stage1-step596057-tokens5001B

## Setup

In [1]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [2]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.9/239.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 151.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4

In [1]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [2]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [3]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Cloning into '/content/tmlr'...
remote: Enumerating objects: 2114, done.
remote: Counting objects: 100% (526/526), done.
remote: Compressing objects: 100% (426/426), done.
remote: Total 2114 (delta 314), reused 293 (delta 93), pack-reused 1588 (from 1)
Receiving objects: 100% (2114/2114), 20.63 MiB | 3.42 MiB/s, done.
Resolving deltas: 100% (1077/1077), done.
Project root: /content/tmlr


In [7]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")

import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [8]:
import src
from src.olmo_config import OLMO_REVISIONS
from src.tl217_olmo2_adapter import load_olmo2_tl217

In [21]:
# Cell 5 - Model name variable
model_name = "OLMo-2-0425-1B"
revision = OLMO_REVISIONS[model_name]

In [22]:
# Cell 6: Load Model
model = load_olmo2_tl217(f"allenai/{model_name}", device=device, revision=revision)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/956M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

OLMo 2 TL2.17 validation
  max |logit difference|:  3.33786e-05
  mean |logit difference|: 3.76754e-06
  top-1 match at every position: True
Loaded pretrained model allenai/OLMo-2-0425-1B into TransformerLens 2.17.0
Layers: 16
Heads: 16
Hidden size: 2048
Params: 1485.3M


In [23]:
# Binding Battery — Loads compound pairs from YAML, measures attention from the
# second constituent back to the first across every layer and head, and saves
# per-domain CSVs plus a run manifest.
from pathlib import Path
import torch
import yaml
import pandas as pd
from src.manifest import write_binding_manifest


def find_token_index(tokens, target):
    """
    Find the index of a target word in a list of tokens.
    Handles leading-space tokenization (e.g., ' screen' for 'screen')
    and subword splits (e.g., 'keyboard' -> ['Key', 'board']).

    For multi-token matches, returns the LAST subtoken index — that's
    where the composed representation lives after the model processes
    the subword sequence.

    Returns the index or None if not found.
    """
    target_lower = target.lower()

    # Pass 1: exact single-token match (stripped of whitespace)
    for i, tok in enumerate(tokens):
        if tok.strip().lower() == target_lower:
            return i

    # Pass 2: multi-token match — concatenate adjacent tokens
    for start in range(len(tokens)):
        concat = ""
        for end in range(start, len(tokens)):
            concat += tokens[end].strip().lower()
            if concat == target_lower:
                # Return the LAST token in the span
                return end
            if len(concat) > len(target_lower):
                break

    return None


prompt_files = [
    'control.yaml',
    'accessibility.yaml',
    'medical.yaml',
    'legal.yaml',
    'finance.yaml',
]

all_results = []
domain_counts = {}
unresolved = []

output_dir = PROJECT_ROOT / 'results' / 'binding' / 'olmo' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

for prompts_file in prompt_files:
    domain = Path(prompts_file).stem
    prompts_path = PROJECT_ROOT / 'data' / 'binding' / prompts_file
    with open(prompts_path, 'r') as f:
        templates = yaml.safe_load(f)
    compounds = templates['compounds']
    print(f"\n--- Running {domain}: {len(compounds)} compounds ---")

    results = []
    for i, case in enumerate(compounds):
        print(f"\r  {i+1}/{len(compounds)}", end="")
        name = case['name']
        word1, word2, prompt = case['word1'], case['word2'], case['prompt']

        tokens = model.to_str_tokens(prompt)
        idx1 = find_token_index(tokens, word1)
        idx2 = find_token_index(tokens, word2)

        if idx1 is None or idx2 is None:
            unresolved.append((domain, name, word1, word2))
            print(f"\n  WARNING: could not locate '{word1}' / '{word2}' in {name}")
            continue

        # word2 attends back to word1 (e.g. "reader" -> "screen")
        target_idx = max(idx1, idx2)   # later token
        source_idx = min(idx1, idx2)   # earlier token

        # Only attention patterns are read — caching every activation wastes
        # a lot of memory on the larger models.
        with torch.no_grad():
            _, cache = model.run_with_cache(
                prompt, names_filter=lambda n: n.endswith("pattern")
            )

        for layer in range(model.cfg.n_layers):
            attention = cache["pattern", layer]   # [batch, heads, seq, seq]
            for head in range(model.cfg.n_heads):
                score = attention[0, head, target_idx, source_idx].item()
                results.append({
                    'compound': name,
                    'layer': layer,
                    'head': head,
                    'binding_score': round(score, 4),
                    'word1': word1,
                    'word2': word2,
                    'prompt': prompt,
                    'tokens': str(tokens),
                    'word1_idx': source_idx,
                    'word2_idx': target_idx,
                    'domain': domain,
                    'model': model_name,
                })

        del cache
    print()

    domain_df = pd.DataFrame(results)
    filename = f'{model_name}-{domain}.csv'
    domain_df.to_csv(output_dir / filename, index=False)
    all_results.append(domain_df)

    domain_counts[domain] = {
        'expected': len(compounds) * model.cfg.n_layers * model.cfg.n_heads,
        'written': len(domain_df),
        'file': filename,
    }
    print(f"  → {filename}  ({len(domain_df)} rows)")

results_df = pd.concat(all_results, ignore_index=True)

write_binding_manifest(
    PROJECT_ROOT, output_dir, model_name, model,
    domain_counts, results_df, prompt_files,
)

if unresolved:
    print(f"\n⚠️  {len(unresolved)} compounds unresolved (skipped):")
    for d, n, w1, w2 in unresolved:
        print(f"    {d}/{n}: '{w1}' / '{w2}'")

print(f"\nSaved {len(results_df)} rows across {len(all_results)} domains → {output_dir}")



--- Running control: 44 compounds ---
  44/44
  → OLMo-2-0425-1B-control.csv  (11264 rows)

--- Running accessibility: 53 compounds ---
  53/53
  → OLMo-2-0425-1B-accessibility.csv  (13568 rows)

--- Running medical: 42 compounds ---
  42/42
  → OLMo-2-0425-1B-medical.csv  (10752 rows)

--- Running legal: 44 compounds ---
  44/44
  → OLMo-2-0425-1B-legal.csv  (11264 rows)

--- Running finance: 44 compounds ---
  44/44
  → OLMo-2-0425-1B-finance.csv  (11264 rows)

Saved 58112 rows across 5 domains → /content/tmlr/results/binding/olmo/OLMo-2-0425-1B


In [24]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "binding results: {model_name}"
!git push

[main cc6a335] binding results: OLMo-2-0425-1B
 6 files changed, 58157 insertions(+)
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-accessibility.csv
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-binding.md
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-control.csv
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-finance.csv
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-legal.csv
 create mode 100644 results/binding/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-medical.csv
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 12 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (12/12), 373.43 KiB | 3.16 MiB/s, done.
Total 12 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/trishasalas/tmlr.git
   1f4f0d3..cc6a335  main -> main


### Delete Model & Clear Cache

In [25]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared — GPU: 54.9GB allocated
